# Pipeline Du Bao Doanh Thu & Gia Von (MLForecast + LightGBM Ensemble)

**Tac gia:** Senior Data Scientist ? Datathon 2026  
**Muc tieu:** Du bao `Revenue` va `COGS` cho 548 ngay (01/01/2023 -> 01/07/2024) bang `mlforecast` (Nixtla) + LightGBM ensemble, co validation-calibrated blend de toi uu MAE.

---
**Kien truc tong quan:**
1. Tien xu ly du lieu -> Long Format va them `is_covid`
2. Khai bao CONFIG tap trung
3. Feature Engineering: calendar, su kien duong lich, COVID regime
4. Huan luyen LightGBM ensemble bang MLForecast
5. Hoc blend weight + median bias tren validation horizon
6. Du bao cac kich ban future COVID (`non_covid`, `covid`, `auto`, `blend`)
7. Xuat `submission.csv` va phan tich SHAP

8. Phuong an 3: Prophet changepoint tai `2019-01-01` de so sanh kha nang bat trend gay.


In [ ]:
%pip install shap
%pip install lightgbm
%pip install mlforecast
%pip install window_ops
%pip install prophet


In [ ]:
# ============================================================
# BUOC 0: IMPORT THU VIEN
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import shap

from lightgbm import LGBMRegressor
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean, RollingMax, RollingMin
from window_ops.rolling import rolling_mean, rolling_max, rolling_min

print("? Import thanh cong.")
print(f"   pandas  : {pd.__version__}")
print(f"   lightgbm: {__import__('lightgbm').__version__}")
print(f"   mlforecast: {__import__('mlforecast').__version__}")


## ⚙️ BƯỚC 1 — CONFIG THAM SỐ

> **TÙY CHỈNH THAM SỐ TẠI ĐÂY** — Toàn bộ hyperparameters, lags, rolling windows và đường dẫn file đều được tập trung trong block `CONFIG` dưới đây. Không cần chỉnh sửa code ở các bước sau.

In [ ]:
# ============================================================
# CONFIG - TOAN BO THAM SO CHINH CUA PIPELINE
# ============================================================

CONFIG = {
    # --- I/O ---
    "input_path": "../data/raw/sales.csv",
    "output_dir": "../data/output",
    "submission_filename": "submission.csv",
    "shap_plot_filename": "shap_summary.png",

    # --- Forecast setup ---
    "forecast_horizon": 548,
    "forecast_start": "2023-01-01",
    "train_ratio": 0.70,
    "date_col": "Date",
    "target_cols": ["Revenue", "COGS"],
    "primary_model_name": "lgbm_base",
    "optimized_model_col": "Optimized",

    # --- COVID regime feature ---
    # Theo yeu cau: giai doan 2019-2022 duoc gan is_covid=1;
    # cac ngay truoc 2019 la 0. Neu muon chinh lai bien 2019, chi can doi covid_start.
    "covid_feature_col": "is_covid",
    "covid_start": "2019-01-01",
    "covid_end": "2022-12-31",
    # Du bao tuong lai co 4 mode:
    #   "non_covid" -> is_covid=0 cho toan bo horizon
    #   "covid"     -> is_covid=1 cho toan bo horizon
    #   "auto"      -> dung rule ngay thang trong covid_start/covid_end
    #   "blend"     -> chay ca non_covid va covid, roi tron theo forecast_covid_probability
    "forecast_covid_mode": "blend",
    "forecast_covid_probability": 0.50,

    # --- Week-of-year power features ---
    # Tang suc nang cho pattern theo tuan: tuan nao thuong la dinh nam, tuan nao ramp-up/ramp-down.
    "week_profile_top_n": 6,
    "week_profile_recent_years": 3,
    "week_growth_clip": (0.80, 1.35),
    "weekofyear_harmonics": [1, 2, 3, 4],
    "use_weekofyear_onehot": True,
    "weekofyear_onehot_weeks": list(range(1, 54)),

    # --- LightGBM ensemble ---
    # lgbm_base giu lai bo tham so hien co; cac model con lai tao da dang de giam MAE qua blend.
    "lgbm_ensemble_params": {
        "lgbm_base": {
            "n_estimators": 1402,
            "learning_rate": 0.03759155708560107,
            "max_depth": 7,
            "num_leaves": 365,
            "min_child_samples": 100,
            "subsample": 0.688363529336043,
            "colsample_bytree": 0.8975121712165626,
            "reg_alpha": 0.06536060672182367,
            "reg_lambda": 0.049218052992262784,
            "random_state": 2404,
            "n_jobs": -1,
            "verbose": -1,
        },
        "lgbm_shallow": {
            "n_estimators": 900,
            "learning_rate": 0.04,
            "max_depth": 4,
            "num_leaves": 31,
            "min_child_samples": 60,
            "subsample": 0.85,
            "colsample_bytree": 0.85,
            "reg_alpha": 0.06536060672182367,
            "reg_lambda": 1.0,
            "random_state": 2404,
            "n_jobs": -1,
            "verbose": -1,
        },
        "lgbm_deep": {
            "n_estimators": 800,
            "learning_rate": 0.06,
            "max_depth": 8,
            "num_leaves": 120,
            "min_child_samples": 30,
            "subsample": 0.80,
            "colsample_bytree": 0.85,
            "reg_alpha": 0.01,
            "reg_lambda": 0.10,
            "random_state": 2404,
            "n_jobs": -1,
            "verbose": -1,
        },
        "lgbm_conservative": {
            "n_estimators": 1500,
            "learning_rate": 0.025,
            "max_depth": 5,
            "num_leaves": 45,
            "min_child_samples": 100,
            "subsample": 0.90,
            "colsample_bytree": 0.90,
            "reg_alpha": 0.10,
            "reg_lambda": 2.0,
            "random_state": 2404,
            "n_jobs": -1,
            "verbose": -1,
        },
    },

    # --- Blend/calibration tren validation horizon ---
    "blend_random_search_n": 5000,
    "blend_dirichlet_alpha": 0.8,
    "blend_random_state": 2404,
    "apply_median_bias": True,
    "clip_negative_predictions": True,

    # --- Lags ---
    "lags": [7, 14, 30, 90, 180, 354, 365],

    # --- Rolling transforms ---
    "rolling_windows": {
        "mean": {"window_sizes": [7, 30], "min_samples": 1},
        "max":  {"window_sizes": [7, 30], "min_samples": 1},
        "min":  {"window_sizes": [7, 30], "min_samples": 1},
    },

    # --- Date features ---
    "date_features": ["dayofweek", "month", "day", "dayofyear"],

    # --- Gregorian peak dates ---
    "gregorian_peaks": [
        {"months": [2],  "days": range(12, 15)},
        {"months": [3],  "days": range(6,  9)},
        {"months": [4],  "days": range(28, 31)},
        {"months": [5],  "days": [1]},
        {"months": [9],  "days": [1, 2]},
        {"months": [12], "days": range(20, 32)},
    ],
}

# Backward-compatible alias for cells that expect one LightGBM param dict.
CONFIG["lgbm_params"] = CONFIG["lgbm_ensemble_params"][CONFIG["primary_model_name"]]

os.makedirs(CONFIG["output_dir"], exist_ok=True)

print("? CONFIG da duoc nap.")
print(f"   Horizon       : {CONFIG['forecast_horizon']} ngay")
print(f"   Train ratio    : {CONFIG['train_ratio']*100:.0f}% / Test ratio: {(1-CONFIG['train_ratio'])*100:.0f}%")
print(f"   Lags           : {CONFIG['lags']}")
print(f"   Rolling windows: {list(CONFIG['rolling_windows']['mean']['window_sizes'])}")
print(f"   Models         : {list(CONFIG['lgbm_ensemble_params'].keys())}")
print(f"   COVID mode     : {CONFIG['forecast_covid_mode']} (p={CONFIG['forecast_covid_probability']:.2f})")


## 📥 BƯỚC 2 — TIỀN XỬ LÝ DỮ LIỆU

Đọc `sales.csv` và chuyển đổi từ định dạng **Wide** (Revenue, COGS là các cột) sang định dạng **Long** bắt buộc của `mlforecast`:

| `unique_id` | `ds`       | `y`   |
|-------------|------------|-------|
| `Revenue`   | 2020-01-01 | 123.4 |
| `COGS`      | 2020-01-01 | 89.2  |

> `unique_id`: định danh time series (Revenue hoặc COGS)  
> `ds`: cột datetime  
> `y`: giá trị cần dự báo

In [ ]:
# ============================================================
# BUOC 2: DOC, THEM is_covid & CHUYEN SANG LONG FORMAT
# ============================================================

def covid_mask(dates: pd.Series | pd.DatetimeIndex) -> np.ndarray:
    """Tra ve mask 0/1 cho giai doan COVID theo CONFIG."""
    date_index = pd.to_datetime(dates)
    start = pd.Timestamp(CONFIG["covid_start"])
    end = pd.Timestamp(CONFIG["covid_end"])
    return ((date_index >= start) & (date_index <= end)).astype(np.int8)


def add_covid_feature(df: pd.DataFrame, date_col: str = "ds", value=None) -> pd.DataFrame:
    """Them cot is_covid. Neu value la 0/1 thi gan cung mot kich ban cho toan bo df."""
    out = df.copy()
    col = CONFIG["covid_feature_col"]
    if value is None or value == "auto":
        out[col] = covid_mask(out[date_col]).astype(np.int8)
    else:
        out[col] = np.int8(value)
    return out


def make_future_exog(unique_ids, future_dates, covid_value="auto", weekly_profile=None) -> pd.DataFrame:
    """Tao future exogenous frame cho MLForecast.predict(X_df=...)."""
    frames = []
    future_dates = pd.to_datetime(pd.Index(future_dates))
    for uid in unique_ids:
        frame = pd.DataFrame({"unique_id": uid, "ds": future_dates})
        frame = add_covid_feature(frame, date_col="ds", value=covid_value)
        frames.append(frame)

    future = pd.concat(frames, ignore_index=True)

    # attach_weekly_profile_features duoc dinh nghia o BUOC 3.
    if weekly_profile is not None:
        future = attach_weekly_profile_features(future, weekly_profile)

    return future


def load_and_melt(filepath: str, date_col: str, target_cols: list) -> pd.DataFrame:
    """
    Doc sales.csv, them feature is_covid va chuyen Wide -> Long format cho mlforecast.
    Output co cot: unique_id, ds, y, is_covid.
    """
    df_wide = pd.read_csv(filepath, parse_dates=[date_col])
    df_wide = df_wide.rename(columns={date_col: "ds"})
    df_wide = add_covid_feature(df_wide, date_col="ds")

    missing = df_wide[target_cols].isnull().sum()
    if missing.any():
        print(f"??  Phat hien gia tri NULL:\n{missing[missing > 0]}")
        df_wide[target_cols] = df_wide[target_cols].interpolate(method="linear")

    df_long = df_wide.melt(
        id_vars=["ds", CONFIG["covid_feature_col"]],
        value_vars=target_cols,
        var_name="unique_id",
        value_name="y",
    )

    df_long = df_long.sort_values(["unique_id", "ds"]).reset_index(drop=True)
    return df_long


# --- Thuc thi ---
df_long = load_and_melt(
    filepath=CONFIG["input_path"],
    date_col=CONFIG["date_col"],
    target_cols=CONFIG["target_cols"],
)

covid_summary = (
    df_long.drop_duplicates("ds")
    .groupby(CONFIG["covid_feature_col"])["ds"]
    .agg(["min", "max", "count"])
)

print("? Da tai, them is_covid va chuyen sang Long Format.")
print(f"   Shape         : {df_long.shape}")
print(f"   unique_id     : {df_long['unique_id'].unique().tolist()}")
print(f"   Khoang thoi gian: {df_long['ds'].min().date()} -> {df_long['ds'].max().date()}")
print(f"   So ngay/series: {df_long.groupby('unique_id')['ds'].count().to_dict()}")
print("\nCOVID feature summary theo ngay:")
display(covid_summary)
df_long.head(4)


In [ ]:
# ============================================================
# BƯỚC 2.5: PHÂN CHIA DỮ LIỆU — TRAIN / TEST (70% / 30%)
# ============================================================
# NGUYÊN TẮC QUAN TRỌNG trong Time Series Forecasting:
#   - KHÔNG split ngẫu nhiên (random split sẽ gây data leakage từ tương lai).
#   - Luôn split theo thứ tự thời gian (temporal split):
#       70% ngày đầu tiên → tập Train (huấn luyện)
#       30% ngày cuối cùng → tập Test  (đánh giá độ chính xác)

def temporal_split(df_long: pd.DataFrame, train_ratio: float) -> tuple:
    """
    Phân chia Long Format DataFrame theo thứ tự thời gian (temporal split).

    Parameters
    ----------
    df_long     : DataFrame Long Format (unique_id, ds, y)
    train_ratio : tỷ lệ dữ liệu dùng để huấn luyện (VD: 0.70)

    Returns
    -------
    (df_train, df_test, cutoff_date)
    """
    # Lấy tất cả ngày duy nhất, sắp xếp tăng dần
    all_dates = np.sort(df_long["ds"].unique())
    n_test = 548
    # n_train   = int(len(all_dates) * train_ratio)
    n_train = len(all_dates) - n_test

    # Ngày cuối cùng thuộc tập Train
    cutoff = pd.Timestamp(all_dates[n_train - 1])

    df_train = df_long[df_long["ds"] <= cutoff].reset_index(drop=True)
    df_test  = df_long[df_long["ds"] >  cutoff].reset_index(drop=True)

    return df_train, df_test, cutoff


# --- Thực thi split ---
df_train, df_test, cutoff_date = temporal_split(
    df_long,
    train_ratio=CONFIG["train_ratio"],
)

n_total = df_long["ds"].nunique()
n_train = df_train["ds"].nunique()
n_test  = df_test["ds"].nunique()

print("✅ Phân chia Train / Test hoàn thành.")
print(f"   Tổng số ngày   : {n_total}")
print(f"   ├─ Train ({CONFIG['train_ratio']*100:.0f}%): {n_train} ngày  "
      f"({df_train['ds'].min().date()} → {df_train['ds'].max().date()})")
print(f"   └─ Test  ({(1-CONFIG['train_ratio'])*100:.0f}%): {n_test} ngày  "
      f"({df_test['ds'].min().date()} → {df_test['ds'].max().date()})")
print(f"   Cutoff date    : {cutoff_date.date()}")

## 🗓️ BƯỚC 3 — FEATURE ENGINEERING: SỰ KIỆN DƯƠNG LỊCH CỐ ĐỊNH

Hàm `add_gregorian_peaks(df)` được thiết kế để nhận diện các **dải ngày Dương lịch** có tính chu kỳ mua sắm cao. Đây là các sự kiện **có thể dự đoán trước** (deterministic), không yêu cầu dữ liệu ngoài.

Ví dụ:
- **12-14/02**: Hiệu ứng Valentine / giai đoạn sát Tết Dương lịch
- **20-31/12**: Mùa mua sắm cuối năm (Christmas season, New Year prep)

In [ ]:
# ============================================================
# BUOC 3: FEATURE ENGINEERING - WEEK-OF-YEAR + YEARLY PEAK TREND
# ============================================================
# Muc tieu: timing cua peak do week-of-year quyet dinh, con do cao cua peak duoc bo sung
# bang recent-year / same-week / annual-peak trend de model hieu xu huong 2020 -> 2021 -> 2022.

def _to_datetime_index(dates) -> pd.DatetimeIndex:
    return pd.DatetimeIndex(pd.to_datetime(dates))


def iso_week_array(dates) -> np.ndarray:
    return _to_datetime_index(dates).isocalendar().week.astype(np.int16).to_numpy()


def iso_year_array(dates) -> np.ndarray:
    return _to_datetime_index(dates).isocalendar().year.astype(np.int16).to_numpy()


def circular_week_distance(week_values, peak_week_values) -> np.ndarray:
    """Khoang cach vong tron giua 2 ISO weeks trong nam 53 tuan."""
    diff = np.abs(np.asarray(week_values, dtype=float) - np.asarray(peak_week_values, dtype=float))
    return np.minimum(diff, 53 - diff)


def year(dates: pd.DatetimeIndex) -> pd.Series:
    return pd.Series(dates.year.astype(np.int16), index=range(len(dates)), name="year")


def weekofyear(dates: pd.DatetimeIndex) -> pd.Series:
    week = dates.isocalendar().week.astype(np.int16).to_numpy()
    return pd.Series(week, index=range(len(dates)), name="weekofyear")


def quarter(dates: pd.DatetimeIndex) -> pd.Series:
    return pd.Series(dates.quarter.astype(np.int8), index=range(len(dates)), name="quarter")


def is_weekend(dates: pd.DatetimeIndex) -> pd.Series:
    return pd.Series((dates.dayofweek >= 5).astype(np.int8), index=range(len(dates)), name="is_weekend")


def is_gregorian_peak(dates: pd.DatetimeIndex) -> pd.Series:
    """1 neu ngay nam trong cac cum su kien duong lich co chu ky mua sam cao."""
    day = dates.day.to_numpy()
    month = dates.month.to_numpy()
    mask = np.zeros(len(dates), dtype=np.int8)

    for peak_rule in CONFIG["gregorian_peaks"]:
        in_month = np.isin(month, list(peak_rule["months"]))
        in_day = np.isin(day, list(peak_rule["days"]))
        mask |= (in_month & in_day).astype(np.int8)

    return pd.Series(mask, index=range(len(dates)), name="is_gregorian_peak")


def build_weekly_profile(reference_df: pd.DataFrame) -> pd.DataFrame:
    """
    Hoc ho so mua vu theo ISO week tu reference_df.

    Hai mat duoc toi uu rieng:
    1. Timing: week rank, peak frequency, top-week frequency, proximity toi peak week.
    2. Gia/bi?n ??: recent same-week mean, last-year same-week mean, YoY growth cua cung tuan,
       annual peak growth va projected next-week level.

    Validation dung df_train lam reference de tranh nhin len df_test.
    Final forecast dung toan bo df_long vi do la lich su da biet.
    """
    tmp = reference_df[["unique_id", "ds", "y"]].copy()
    tmp["woy_week"] = iso_week_array(tmp["ds"])
    tmp["iso_year"] = iso_year_array(tmp["ds"])

    week_year = (
        tmp.groupby(["unique_id", "iso_year", "woy_week"])["y"]
        .agg(woy_year_week_mean="mean", woy_year_week_median="median", woy_year_week_max="max", woy_year_week_days="size")
        .reset_index()
        .sort_values(["unique_id", "woy_week", "iso_year"])
    )

    weekly = (
        tmp.groupby(["unique_id", "woy_week"])["y"]
        .agg(
            woy_profile_mean="mean",
            woy_profile_median="median",
            woy_profile_std="std",
            woy_profile_count="count",
        )
        .reset_index()
    )

    target_mean = tmp.groupby("unique_id")["y"].mean().rename("woy_target_mean").reset_index()
    full_index = pd.MultiIndex.from_product(
        [tmp["unique_id"].unique(), range(1, 54)],
        names=["unique_id", "woy_week"],
    ).to_frame(index=False)

    profile = full_index.merge(weekly, on=["unique_id", "woy_week"], how="left")
    profile = profile.merge(target_mean, on="unique_id", how="left")

    profile["woy_profile_mean"] = profile["woy_profile_mean"].fillna(profile["woy_target_mean"])
    profile["woy_profile_median"] = profile["woy_profile_median"].fillna(profile["woy_target_mean"])
    profile["woy_profile_std"] = profile["woy_profile_std"].fillna(0.0)
    profile["woy_profile_count"] = profile["woy_profile_count"].fillna(0).astype(np.int16)
    profile["woy_profile_ratio"] = profile["woy_profile_mean"] / profile["woy_target_mean"].replace(0, np.nan)
    profile["woy_profile_ratio"] = profile["woy_profile_ratio"].replace([np.inf, -np.inf], np.nan).fillna(1.0)
    profile["woy_profile_rank_pct"] = profile.groupby("unique_id")["woy_profile_mean"].rank(pct=True)

    # --- Recent-year amplitude layer: dung 3 nam gan nhat de bat xu huong peak dang tang. ---
    recent_years = CONFIG["week_profile_recent_years"]
    low_growth, high_growth = CONFIG["week_growth_clip"]
    max_year = week_year.groupby("unique_id")["iso_year"].transform("max")
    recent_week_year = week_year[week_year["iso_year"] >= max_year - recent_years + 1].copy()

    recent_stats = (
        recent_week_year.groupby(["unique_id", "woy_week"])["woy_year_week_mean"]
        .agg(
            woy_recent_mean="mean",
            woy_recent_median="median",
            woy_recent_max="max",
            woy_recent_count="count",
        )
        .reset_index()
    )
    last_week = (
        week_year.sort_values(["unique_id", "woy_week", "iso_year"])
        .groupby(["unique_id", "woy_week"])
        .tail(1)[["unique_id", "woy_week", "woy_year_week_mean", "iso_year"]]
        .rename(columns={"woy_year_week_mean": "woy_last_year_mean", "iso_year": "woy_last_observed_year"})
    )

    week_year["woy_prev_year_week_mean"] = week_year.groupby(["unique_id", "woy_week"])["woy_year_week_mean"].shift(1)
    week_year["woy_same_week_yoy_growth"] = week_year["woy_year_week_mean"] / week_year["woy_prev_year_week_mean"].replace(0, np.nan)
    week_year["woy_same_week_yoy_growth"] = week_year["woy_same_week_yoy_growth"].replace([np.inf, -np.inf], np.nan)
    recent_growth = week_year[week_year["iso_year"] >= max_year - recent_years + 1]
    growth_stats = (
        recent_growth.groupby(["unique_id", "woy_week"])["woy_same_week_yoy_growth"]
        .median()
        .clip(low_growth, high_growth)
        .fillna(1.0)
        .rename("woy_same_week_growth")
        .reset_index()
    )

    profile = profile.merge(recent_stats, on=["unique_id", "woy_week"], how="left")
    profile = profile.merge(last_week, on=["unique_id", "woy_week"], how="left")
    profile = profile.merge(growth_stats, on=["unique_id", "woy_week"], how="left")

    for col in ["woy_recent_mean", "woy_recent_median", "woy_recent_max", "woy_last_year_mean"]:
        profile[col] = profile[col].fillna(profile["woy_profile_mean"])
    profile["woy_recent_count"] = profile["woy_recent_count"].fillna(0).astype(np.int16)
    profile["woy_last_observed_year"] = profile["woy_last_observed_year"].fillna(tmp["iso_year"].max()).astype(np.int16)
    profile["woy_same_week_growth"] = profile["woy_same_week_growth"].fillna(1.0).clip(low_growth, high_growth)
    profile["woy_recent_to_all_ratio"] = profile["woy_recent_mean"] / profile["woy_profile_mean"].replace(0, np.nan)
    profile["woy_last_to_all_ratio"] = profile["woy_last_year_mean"] / profile["woy_profile_mean"].replace(0, np.nan)
    profile["woy_projected_next_mean"] = profile["woy_last_year_mean"] * profile["woy_same_week_growth"]
    profile["woy_projected_to_all_ratio"] = profile["woy_projected_next_mean"] / profile["woy_profile_mean"].replace(0, np.nan)
    ratio_cols = ["woy_recent_to_all_ratio", "woy_last_to_all_ratio", "woy_projected_to_all_ratio"]
    profile[ratio_cols] = profile[ratio_cols].replace([np.inf, -np.inf], np.nan).fillna(1.0).clip(0.50, 2.50)

    # --- Peak timing layer: tan suat tuan do la dinh nam/top-3 nam. ---
    annual_week = (
        tmp.groupby(["unique_id", "iso_year", "woy_week"])["y"]
        .sum()
        .reset_index(name="woy_year_week_total")
    )
    annual_week["woy_rank_in_year"] = annual_week.groupby(["unique_id", "iso_year"])["woy_year_week_total"].rank(
        method="min", ascending=False
    )
    peak_freq = (
        annual_week.assign(
            woy_peak_hit=(annual_week["woy_rank_in_year"] == 1).astype(float),
            woy_top3_hit=(annual_week["woy_rank_in_year"] <= 3).astype(float),
        )
        .groupby(["unique_id", "woy_week"])
        .agg(
            woy_peak_frequency=("woy_peak_hit", "mean"),
            woy_top3_frequency=("woy_top3_hit", "mean"),
        )
        .reset_index()
    )

    profile = profile.merge(peak_freq, on=["unique_id", "woy_week"], how="left")
    profile[["woy_peak_frequency", "woy_top3_frequency"]] = profile[["woy_peak_frequency", "woy_top3_frequency"]].fillna(0.0)
    profile["woy_peak_score"] = (
        0.45 * profile["woy_profile_rank_pct"]
        + 0.35 * profile["woy_top3_frequency"]
        + 0.20 * profile["woy_peak_frequency"]
    )

    top_n = CONFIG["week_profile_top_n"]
    profile["woy_mean_rank_desc"] = profile.groupby("unique_id")["woy_profile_mean"].rank(method="first", ascending=False)
    profile["woy_peak_rank_desc"] = profile.groupby("unique_id")["woy_peak_score"].rank(method="first", ascending=False)
    profile["woy_is_top_mean_week"] = (profile["woy_mean_rank_desc"] <= top_n).astype(np.int8)
    profile["woy_is_peak_candidate_week"] = (profile["woy_peak_rank_desc"] <= top_n).astype(np.int8)

    peak_week_map = (
        profile.sort_values(["unique_id", "woy_peak_score", "woy_profile_mean"], ascending=[True, False, False])
        .groupby("unique_id")
        .first()["woy_week"]
        .to_dict()
    )
    profile["woy_reference_peak_week"] = profile["unique_id"].map(peak_week_map).astype(np.int16)
    profile["woy_peak_distance"] = circular_week_distance(profile["woy_week"], profile["woy_reference_peak_week"]).astype(float)
    profile["woy_peak_proximity"] = 1.0 - (profile["woy_peak_distance"] / 26.5)
    profile["woy_peak_proximity"] = profile["woy_peak_proximity"].clip(0.0, 1.0)

    # --- Annual peak growth layer: mot tin hieu chung cho do cao dinh moi nam. ---
    annual_peak = (
        annual_week.sort_values(["unique_id", "iso_year", "woy_year_week_total"], ascending=[True, True, False])
        .groupby(["unique_id", "iso_year"])
        .head(1)
        .sort_values(["unique_id", "iso_year"])
        .copy()
    )
    annual_peak["prev_peak_total"] = annual_peak.groupby("unique_id")["woy_year_week_total"].shift(1)
    annual_peak["annual_peak_yoy_growth"] = annual_peak["woy_year_week_total"] / annual_peak["prev_peak_total"].replace(0, np.nan)
    recent_annual_peak = annual_peak[
        annual_peak["iso_year"] >= annual_peak.groupby("unique_id")["iso_year"].transform("max") - recent_years + 1
    ].copy()
    annual_growth = (
        recent_annual_peak.groupby("unique_id")["annual_peak_yoy_growth"]
        .median()
        .clip(low_growth, high_growth)
        .fillna(1.0)
        .rename("woy_annual_peak_growth")
        .reset_index()
    )
    latest_peak = (
        annual_peak.sort_values(["unique_id", "iso_year"])
        .groupby("unique_id")
        .tail(1)[["unique_id", "iso_year", "woy_year_week_total"]]
        .rename(columns={"iso_year": "woy_last_peak_year", "woy_year_week_total": "woy_last_annual_peak_total"})
    )
    profile = profile.merge(annual_growth, on="unique_id", how="left")
    profile = profile.merge(latest_peak, on="unique_id", how="left")
    profile["woy_annual_peak_growth"] = profile["woy_annual_peak_growth"].fillna(1.0).clip(low_growth, high_growth)
    profile["woy_expected_next_annual_peak"] = profile["woy_last_annual_peak_total"] * profile["woy_annual_peak_growth"]
    profile["woy_expected_peak_to_current"] = profile["woy_expected_next_annual_peak"] / profile["woy_last_annual_peak_total"].replace(0, np.nan)
    profile["woy_expected_peak_to_current"] = profile["woy_expected_peak_to_current"].replace([np.inf, -np.inf], np.nan).fillna(1.0)

    return profile


def attach_weekly_profile_features(df: pd.DataFrame, weekly_profile: pd.DataFrame) -> pd.DataFrame:
    """Gan weekly profile, cyclic week harmonics va one-hot week vao df/input future."""
    out = df.copy()
    week_values = iso_week_array(out["ds"])
    out["woy_week"] = week_values.astype(np.int16)

    feature_block = {}
    for harmonic in CONFIG["weekofyear_harmonics"]:
        angle = 2 * np.pi * harmonic * week_values / 53.0
        feature_block[f"woy_sin_{harmonic}"] = np.sin(angle).astype(np.float32)
        feature_block[f"woy_cos_{harmonic}"] = np.cos(angle).astype(np.float32)

    if CONFIG["use_weekofyear_onehot"]:
        for week in CONFIG["weekofyear_onehot_weeks"]:
            feature_block[f"woy_oh_{week:02d}"] = (week_values == week).astype(np.int8)

    out = pd.concat([out, pd.DataFrame(feature_block, index=out.index)], axis=1)
    out = out.merge(weekly_profile, on=["unique_id", "woy_week"], how="left")
    return out


ADVANCED_DATE_FEATURES = CONFIG["date_features"] + [
    year,
    weekofyear,
    quarter,
    is_weekend,
    is_gregorian_peak,
]

# Validation profile: chi hoc tu df_train, sau do ap cho df_train va df_test.
WEEKLY_PROFILE_TRAIN = build_weekly_profile(df_train)
df_train_model = attach_weekly_profile_features(df_train, WEEKLY_PROFILE_TRAIN)
df_test_model = attach_weekly_profile_features(df_test, WEEKLY_PROFILE_TRAIN)
WEEKLY_FEATURE_COLS = [c for c in df_train_model.columns if c.startswith("woy_")]

# Kiem tra nhanh feature lich va cac tuan dinh hoc duoc.
_test_idx = pd.date_range("2022-01-01", periods=365, freq="D")
_test_series = is_gregorian_peak(_test_idx)
peak_week_preview = (
    WEEKLY_PROFILE_TRAIN.sort_values(["unique_id", "woy_peak_score"], ascending=[True, False])
    .groupby("unique_id")
    .head(CONFIG["week_profile_top_n"])
    [["unique_id", "woy_week", "woy_profile_ratio", "woy_recent_to_all_ratio", "woy_same_week_growth", "woy_annual_peak_growth", "woy_peak_score"]]
)

print("? Week-of-year + yearly peak trend feature layer da san sang.")
print(f"   Date features       : {CONFIG['date_features'] + ['year', 'weekofyear', 'quarter', 'is_weekend', 'is_gregorian_peak']}")
print(f"   Weekly exog features : {len(WEEKLY_FEATURE_COLS)} cot")
print(f"   Recent years cho gia : {CONFIG['week_profile_recent_years']} nam gan nhat")
print(f"   One-hot weeks       : {CONFIG['use_weekofyear_onehot']} ({len(CONFIG['weekofyear_onehot_weeks'])} cot)")
print(f"   So ngay peak trong nam 2022: {int(_test_series.sum())} / 365")
print(f"   is_covid 2022-01-01: {int(covid_mask(pd.DatetimeIndex(['2022-01-01']))[0])}")
print(f"   is_covid 2023-01-01(auto): {int(covid_mask(pd.DatetimeIndex(['2023-01-01']))[0])}")
print("\nTop weekly peak candidates + amplitude trend hoc tu TRAIN:")
display(peak_week_preview)


## BUOC 4 - HUAN LUYEN & DU BAO (Timing + Peak Amplitude)

Ban nay toi uu 2 mat tach rieng:

1. **Timing cua peak trong nam**: dung `weekofyear`, one-hot 53 tuan, harmonic sin/cos, peak frequency va top-3-week frequency de model biet tuan nao trong nam thuong dat dinh.
2. **Do cao/gia cua peak**: them cac bien trend theo nam de model hieu chuoi peak 2020 -> 2021 -> 2022 dang tang, thay vi chi lay trung binh lich su.

Cac feature bien do moi gom:

- `woy_recent_mean`, `woy_recent_median`, `woy_recent_max`: muc cua cung tuan trong 3 nam gan nhat.
- `woy_last_year_mean`: muc cua cung tuan o nam gan nhat co du lieu.
- `woy_same_week_growth`: YoY growth cua cung week-of-year.
- `woy_projected_next_mean`: uoc luong muc cua cung tuan neu tiep tuc theo recent YoY growth.
- `woy_annual_peak_growth`: toc do tang cua annual peak trong cac nam gan nhat.
- `woy_expected_next_annual_peak`: dinh nam tiep theo du kien theo peak trend.

Validation profile chi hoc tu train, sau do ap cho test horizon de tranh leakage. Final forecast hoc weekly/peak profile tu toan bo lich su 2012-2022.


In [ ]:
# ============================================================
# BUOC 4: SETUP MLFORECAST + LIGHTGBM ENSEMBLE
# ============================================================

# --- 4.1 Xay dung lag_transforms tu CONFIG ---
lag_transforms = {}

rw_cfg = CONFIG["rolling_windows"]
for win in rw_cfg["mean"]["window_sizes"]:
    min_s = rw_cfg["mean"]["min_samples"]
    lag_transforms[win] = [
        RollingMean(window_size=win, min_samples=min_s),
        RollingMax(window_size=win, min_samples=min_s),
        RollingMin(window_size=win, min_samples=min_s),
    ]

print("?? Lag transforms duoc cau hinh:")
for k, v in lag_transforms.items():
    print(f"   lag_{k}: {[type(t).__name__ for t in v]}")


# --- 4.2 Khoi tao LightGBM ensemble ---
def build_lgbm_models() -> dict:
    return {
        name: LGBMRegressor(**params)
        for name, params in CONFIG["lgbm_ensemble_params"].items()
    }


MODEL_COLS = list(CONFIG["lgbm_ensemble_params"].keys())


# --- 4.3 Khoi tao MLForecast ---
# is_covid va cac woy_* cot la dynamic exogenous features.
fcst = MLForecast(
    models=build_lgbm_models(),
    freq="D",
    lags=CONFIG["lags"],
    lag_transforms=lag_transforms,
    date_features=ADVANCED_DATE_FEATURES,
    num_threads=4,
)

print("\n? MLForecast da duoc khoi tao.")
print(f"   Lags         : {CONFIG['lags']}")
print(f"   Model columns: {MODEL_COLS}")
print(f"   Dynamic exog : {CONFIG['covid_feature_col']} + {len(WEEKLY_FEATURE_COLS)} woy_* features")
print(f"   Date features: {CONFIG['date_features'] + ['year', 'weekofyear', 'quarter', 'is_weekend', 'is_gregorian_peak']}")


# --- 4.4 Preprocess: kiem tra feature matrix tren tap Train ---
print("\n? Dang preprocess & tao feature matrix (tren tap Train)...")
X_train_df = fcst.preprocess(df_train_model, static_features=[])

print("? Preprocess hoan thanh.")
print(f"   Feature matrix shape: {X_train_df.shape}")
print(f"   Co is_covid trong feature matrix: {CONFIG['covid_feature_col'] in X_train_df.columns}")
print(f"   Weekly feature count trong matrix: {sum(c.startswith('woy_') for c in X_train_df.columns)}")
print(f"   Tong so feature: {len([c for c in X_train_df.columns if c not in ['unique_id', 'ds', 'y']])}")
X_train_df.head(4)


In [ ]:
# --- 4.5 Huan luyen ensemble tren tap Train ---
print(f"? Dang huan luyen {len(MODEL_COLS)} LightGBM models tren tap Train "
      f"({CONFIG['train_ratio']*100:.0f}% lich su) voi week-of-year profile...")

fcst.fit(df_train_model, static_features=[])

print("? Huan luyen hoan thanh.")
print(f"   Khoang du lieu train: "
      f"{df_train['ds'].min().date()} -> {df_train['ds'].max().date()}")


In [ ]:
# ============================================================
# BUOC 4.5: DANH GIA + HOC BLEND DE GIAM MAE TREN TAP TEST
# ============================================================
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def make_weight_candidates(model_cols: list, n_random: int, alpha: float, seed: int) -> list:
    """Sinh cac bo weight khong am, tong = 1. Them one-hot de khong bao gio te hon model don tot nhat."""
    n_models = len(model_cols)
    rng = np.random.default_rng(seed)
    candidates = [np.eye(n_models)[i] for i in range(n_models)]
    candidates.append(np.ones(n_models) / n_models)
    candidates.extend(rng.dirichlet(np.ones(n_models) * alpha) for _ in range(n_random))
    return candidates


def learn_blend_config(eval_df: pd.DataFrame, model_cols: list) -> dict:
    """Hoc weight + median bias rieng cho Revenue/COGS tren validation horizon."""
    candidates = make_weight_candidates(
        model_cols=model_cols,
        n_random=CONFIG["blend_random_search_n"],
        alpha=CONFIG["blend_dirichlet_alpha"],
        seed=CONFIG["blend_random_state"],
    )
    blend_config = {}

    for uid in CONFIG["target_cols"]:
        mask = eval_df["unique_id"] == uid
        y_true = eval_df.loc[mask, "y"].to_numpy(dtype=float)
        preds = eval_df.loc[mask, model_cols].to_numpy(dtype=float)

        best_mae = np.inf
        best_weights = None
        best_bias = 0.0

        for weights in candidates:
            raw_pred = preds @ weights
            bias = float(np.median(y_true - raw_pred)) if CONFIG["apply_median_bias"] else 0.0
            pred = raw_pred + bias
            if CONFIG["clip_negative_predictions"]:
                pred = np.maximum(pred, 0.0)
            mae = mean_absolute_error(y_true, pred)
            if mae < best_mae:
                best_mae = mae
                best_weights = weights.copy()
                best_bias = bias

        blend_config[uid] = {
            "weights": {col: float(w) for col, w in zip(model_cols, best_weights)},
            "bias": float(best_bias),
            "validation_mae": float(best_mae),
        }

    return blend_config


def apply_blend(forecast_df: pd.DataFrame, blend_config: dict, model_cols: list, output_col: str | None = None) -> pd.DataFrame:
    """Ap dung blend_config vao dataframe du bao long format."""
    output_col = output_col or CONFIG["optimized_model_col"]
    out = forecast_df.copy()
    out[output_col] = np.nan

    for uid, cfg in blend_config.items():
        mask = out["unique_id"] == uid
        pred = np.zeros(mask.sum(), dtype=float)
        for col, weight in cfg["weights"].items():
            pred += out.loc[mask, col].to_numpy(dtype=float) * weight
        pred += cfg["bias"]
        if CONFIG["clip_negative_predictions"]:
            pred = np.maximum(pred, 0.0)
        out.loc[mask, output_col] = pred

    return out


test_horizon = df_test["ds"].nunique()
print(f"? Du bao {test_horizon} ngay de danh gia tren tap Test (30%) voi weekly profile...")

# Validation dung is_covid theo ngay thuc te trong df_test.
eval_future_exog = make_future_exog(
    unique_ids=df_train["unique_id"].unique(),
    future_dates=np.sort(df_test["ds"].unique()),
    covid_value="auto",
    weekly_profile=WEEKLY_PROFILE_TRAIN,
)
forecast_eval_long = fcst.predict(h=test_horizon, X_df=eval_future_exog)

# Ghep ket qua du bao voi gia tri thuc te cua tap Test
eval_df_raw = forecast_eval_long.merge(
    df_test[["unique_id", "ds", "y"]],
    on=["unique_id", "ds"],
    how="inner",
)

BLEND_CONFIG = learn_blend_config(eval_df_raw, MODEL_COLS)
eval_df = apply_blend(eval_df_raw, BLEND_CONFIG, MODEL_COLS)

print(f"\n{'='*72}")
print(f"{'?? KET QUA DANH GIA TREN TAP TEST (30%)':^72}")
print(f"{'='*72}")
print(f"{'Metric':<18} {'Revenue':>19} {'COGS':>19} {'Avg':>12}")
print(f"{'-'*72}")

eval_results = {}
for uid in CONFIG["target_cols"]:
    mask = eval_df["unique_id"] == uid
    y_true = eval_df.loc[mask, "y"].values
    y_pred = eval_df.loc[mask, CONFIG["optimized_model_col"]].values
    safe_y = np.where(np.abs(y_true) < 1e-8, 1e-8, y_true)
    eval_results[uid] = {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "MAPE(%)": np.mean(np.abs((y_true - y_pred) / safe_y)) * 100,
    }

for metric in ["MAE", "RMSE", "R2", "MAPE(%)"]:
    rev = eval_results["Revenue"][metric]
    cogs = eval_results["COGS"][metric]
    avg = (rev + cogs) / 2
    if metric == "R2":
        print(f"{metric:<18} {rev:>19.4f} {cogs:>19.4f} {avg:>12.4f}")
    elif metric == "MAPE(%)":
        print(f"{metric:<18} {rev:>18.2f}% {cogs:>18.2f}% {avg:>11.2f}%")
    else:
        print(f"{metric:<18} {rev:>19,.2f} {cogs:>19,.2f} {avg:>12,.2f}")

print(f"{'='*72}")
print("\nBlend weights da hoc tren validation:")
for uid, cfg in BLEND_CONFIG.items():
    weights_txt = ", ".join(f"{k}={v:.3f}" for k, v in cfg["weights"].items() if v > 1e-3)
    print(f"   {uid:<7} | MAE={cfg['validation_mae']:,.2f} | bias={cfg['bias']:,.2f} | {weights_txt}")

print("\nMAE tung model don le (de so sanh):")
for model_col in MODEL_COLS:
    row = []
    for uid in CONFIG["target_cols"]:
        mask = eval_df_raw["unique_id"] == uid
        row.append(mean_absolute_error(eval_df_raw.loc[mask, "y"], eval_df_raw.loc[mask, model_col]))
    print(f"   {model_col:<18} Revenue={row[0]:>12,.2f} | COGS={row[1]:>12,.2f} | Avg={(row[0]+row[1])/2:>12,.2f}")


In [ ]:
# ============================================================
# BUOC 4.6: TAI HUAN LUYEN TREN TOAN BO DU LIEU & DU BAO SCENARIO
# ============================================================

def combine_forecast_scenarios(non_covid_df: pd.DataFrame, covid_df: pd.DataFrame, covid_probability: float) -> pd.DataFrame:
    """Tron 2 kich ban non-covid/covid theo xac suat covid_probability."""
    p = float(np.clip(covid_probability, 0.0, 1.0))
    out = non_covid_df.copy()
    pred_cols = MODEL_COLS + [CONFIG["optimized_model_col"]]
    for col in pred_cols:
        out[col] = (1.0 - p) * non_covid_df[col].to_numpy(dtype=float) + p * covid_df[col].to_numpy(dtype=float)
    return out


def predict_with_covid_scenario(fcst_obj: MLForecast, h: int, scenario: str) -> pd.DataFrame:
    """Du bao theo mot kich ban is_covid cho future horizon, kem weekly profile hoc tu full history."""
    future_dates = pd.date_range(CONFIG["forecast_start"], periods=h, freq="D")
    if scenario == "non_covid":
        covid_value = 0
    elif scenario == "covid":
        covid_value = 1
    elif scenario == "auto":
        covid_value = "auto"
    else:
        raise ValueError(f"Scenario khong hop le: {scenario}")

    future_exog = make_future_exog(
        unique_ids=df_long["unique_id"].unique(),
        future_dates=future_dates,
        covid_value=covid_value,
        weekly_profile=WEEKLY_PROFILE_FULL,
    )
    raw_forecast = fcst_obj.predict(h=h, X_df=future_exog)
    return apply_blend(raw_forecast, BLEND_CONFIG, MODEL_COLS)


print("? Xay dung weekly profile tren 100% lich su...")
WEEKLY_PROFILE_FULL = build_weekly_profile(df_long)
df_long_model = attach_weekly_profile_features(df_long, WEEKLY_PROFILE_FULL)

full_peak_week_preview = (
    WEEKLY_PROFILE_FULL.sort_values(["unique_id", "woy_peak_score"], ascending=[True, False])
    .groupby("unique_id")
    .head(CONFIG["week_profile_top_n"])
    [[
        "unique_id",
        "woy_week",
        "woy_profile_ratio",
        "woy_recent_to_all_ratio",
        "woy_same_week_growth",
        "woy_annual_peak_growth",
        "woy_peak_score",
    ]]
)

annual_peak_note = (
    df_long.assign(
        iso_year=iso_year_array(df_long["ds"]),
        woy_week=iso_week_array(df_long["ds"]),
    )
    .groupby(["unique_id", "iso_year", "woy_week"])["y"]
    .mean()
    .reset_index(name="weekly_mean")
)
annual_peak_note = (
    annual_peak_note.sort_values(["unique_id", "iso_year", "weekly_mean"], ascending=[True, True, False])
    .groupby(["unique_id", "iso_year"])
    .head(1)
    .sort_values(["unique_id", "iso_year"])
)
annual_peak_note["peak_yoy_growth"] = annual_peak_note.groupby("unique_id")["weekly_mean"].pct_change() + 1
annual_peak_note = annual_peak_note[annual_peak_note["iso_year"].between(2020, 2022)]

print("Top weekly peak candidates hoc tu FULL history:")
display(full_peak_week_preview)
print("\nNOTE - bang nay cho thay peak 2020-2022 theo tung target, dung de model hoc do cao/gia cua dinh:")
display(annual_peak_note)

print("\n? Tai huan luyen LightGBM ensemble tren 100% du lieu lich su...")
fcst.fit(df_long_model, static_features=[])
print("? Tai huan luyen hoan thanh.")
print(f"   Khoang du lieu full: {df_long['ds'].min().date()} -> {df_long['ds'].max().date()}")

print(f"\n? Dang du bao {CONFIG['forecast_horizon']} ngay toi...")
forecast_scenarios = {
    "non_covid": predict_with_covid_scenario(fcst, CONFIG["forecast_horizon"], "non_covid"),
    "covid": predict_with_covid_scenario(fcst, CONFIG["forecast_horizon"], "covid"),
    "auto": predict_with_covid_scenario(fcst, CONFIG["forecast_horizon"], "auto"),
}

mode = CONFIG["forecast_covid_mode"]
if mode == "blend":
    forecast_long = combine_forecast_scenarios(
        forecast_scenarios["non_covid"],
        forecast_scenarios["covid"],
        CONFIG["forecast_covid_probability"],
    )
    forecast_scenarios["blend"] = forecast_long
elif mode in forecast_scenarios:
    forecast_long = forecast_scenarios[mode]
else:
    raise ValueError("forecast_covid_mode phai la: non_covid, covid, auto, hoac blend")

print("? Du bao hoan thanh.")
print(f"   Scenario official     : {mode}")
if mode == "blend":
    print(f"   COVID probability     : {CONFIG['forecast_covid_probability']:.2f}")
print(f"   Shape ket qua (Long)  : {forecast_long.shape}")
print(f"   Ngay dau du bao       : {forecast_long['ds'].min().date()}")
print(f"   Ngay cuoi du bao      : {forecast_long['ds'].max().date()}")
forecast_long.head(6)


## 💾 BƯỚC 5 — XUẤT KẾT QUẢ (`submission.csv`)

Chuyển (Pivot) dữ liệu dự báo từ **Long format** trở lại **Wide format** (`Date`, `Revenue`, `COGS`) và lưu ra file `submission.csv`.

In [ ]:
# ============================================================
# BUOC 5: PIVOT LONG -> WIDE VA LUU submission.csv
# ============================================================

def pivot_to_wide(forecast_long: pd.DataFrame, model_col: str | None = None) -> pd.DataFrame:
    """
    Chuyen ket qua du bao tu Long format (unique_id, ds, prediction)
    sang Wide format (Date, Revenue, COGS) theo yeu cau nop bai.
    """
    model_col = model_col or CONFIG["optimized_model_col"]

    df_wide = forecast_long.pivot(
        index="ds",
        columns="unique_id",
        values=model_col,
    ).reset_index()

    df_wide = df_wide.rename(columns={"ds": "Date"})
    df_wide.columns.name = None

    col_order = ["Date"] + CONFIG["target_cols"]
    df_wide = df_wide[col_order]

    for col in CONFIG["target_cols"]:
        df_wide[col] = df_wide[col].clip(lower=0).round(2)

    return df_wide


# Pivot official forecast ve Wide format
submission = pivot_to_wide(forecast_long, model_col=CONFIG["optimized_model_col"])

assert str(submission["Date"].min().date()) == CONFIG["forecast_start"], (
    f"? Ngay bat dau sai: {submission['Date'].min().date()} (ky vong {CONFIG['forecast_start']})"
)
assert len(submission) == CONFIG["forecast_horizon"], (
    f"? So dong sai: {len(submission)} (ky vong {CONFIG['forecast_horizon']})"
)

# Luu official submission
output_path = os.path.join(CONFIG["output_dir"], CONFIG["submission_filename"])
submission.to_csv(output_path, index=False, date_format="%Y-%m-%d")

# Luu them cac scenario de ban co the nop/so sanh nhanh neu muon doi gia dinh COVID tuong lai.
scenario_outputs = {}
for scenario_name, scenario_forecast in forecast_scenarios.items():
    scenario_submission = pivot_to_wide(scenario_forecast, model_col=CONFIG["optimized_model_col"])
    scenario_path = os.path.join(CONFIG["output_dir"], f"submission_{scenario_name}.csv")
    scenario_submission.to_csv(scenario_path, index=False, date_format="%Y-%m-%d")
    scenario_outputs[scenario_name] = scenario_path

print("? Da luu submission.csv.")
print(f"   Duong dan official : {output_path}")
print(f"   Model column       : {CONFIG['optimized_model_col']}")
print(f"   COVID mode official: {CONFIG['forecast_covid_mode']}")
print(f"   So dong           : {len(submission)}")
print(f"   Ngay dau          : {submission['Date'].min().date()}")
print(f"   Ngay cuoi         : {submission['Date'].max().date()}")
print("\nScenario files:")
for name, path in scenario_outputs.items():
    print(f"   {name:<10}: {path}")

print("\n--- Mau ket qua (5 dong dau) ---")
display(submission.head())
print("\n--- Mau ket qua (5 dong cuoi) ---")
display(submission.tail())


## PHUONG AN 3 - Prophet Changepoints de bat trend gay

Prophet duoc thiet ke rat hop voi bai toan trend bi gay hoac doi regime. O day ta ep Prophet nhan dien ngay bat dau giai doan khung hoang `2019-01-01` nhu mot changepoint cung. Khi do mo hinh co quyen uon lai trend sau moc nay, thay vi co keo mot duong xu huong lien tuc tu 2012 den 2022.

Muc tieu cua nhanh nay:

- Giu seasonal pattern theo nam bang `yearly_seasonality=True`.
- Cho trend linh hoat hon sau changepoint bang `changepoint_prior_scale=0.1`.
- Train rieng Prophet cho `Revenue` va `COGS`.
- Danh gia tren cung validation horizon 548 ngay.
- Xuat them `submission_prophet_changepoint.csv` de so sanh voi submission LightGBM ensemble.

Luu y: day la phuong an so sanh/ensemble candidate, khong ghi de `submission.csv` chinh cua pipeline LightGBM.


In [ ]:
# ============================================================
# PHUONG AN 3: PROPHET CHANGEPOINTS CHO TREND BI GAY
# ============================================================
# Note tieng Viet:
# - LightGBM/MLForecast dang manh o pattern theo week-of-year va lag.
# - Prophet duoc them nhu mot phuong an rieng de bat trend bi gay tu 2019-01-01.
# - File output rieng: submission_prophet_changepoint.csv, khong ghi de submission.csv chinh.

try:
    from prophet import Prophet
except ImportError as exc:
    raise ImportError(
        "Chua cai prophet. Hay chay cell cai dat o dau notebook: `%pip install prophet`, "
        "sau do restart kernel neu can."
    ) from exc

PROPHET_CONFIG = {
    "changepoints": [CONFIG["covid_start"]],   # bat Prophet uon trend tu 2019-01-01
    "changepoint_prior_scale": 0.10,           # trend linh hoat hon sau diem gay
    "yearly_seasonality": True,
    "weekly_seasonality": True,
    "daily_seasonality": False,
    "seasonality_mode": "multiplicative",     # doanh thu/COGS co bien do mua vu thay doi theo level
    "add_covid_regressor": True,               # them is_covid nhu regressor de hoc level shift
    "interval_width": 0.80,
}


def prepare_prophet_df(df_long_input: pd.DataFrame, target_id: str) -> pd.DataFrame:
    """Chuyen long format cua mot target sang format Prophet: ds, y, is_covid."""
    cols = ["ds", "y", CONFIG["covid_feature_col"]]
    out = (
        df_long_input[df_long_input["unique_id"] == target_id][cols]
        .sort_values("ds")
        .reset_index(drop=True)
    )
    return out


def build_prophet_model() -> Prophet:
    """Khoi tao Prophet voi changepoint bat buoc tai ngay bat dau khung hoang."""
    model = Prophet(
        changepoints=PROPHET_CONFIG["changepoints"],
        changepoint_prior_scale=PROPHET_CONFIG["changepoint_prior_scale"],
        yearly_seasonality=PROPHET_CONFIG["yearly_seasonality"],
        weekly_seasonality=PROPHET_CONFIG["weekly_seasonality"],
        daily_seasonality=PROPHET_CONFIG["daily_seasonality"],
        seasonality_mode=PROPHET_CONFIG["seasonality_mode"],
        interval_width=PROPHET_CONFIG["interval_width"],
    )
    if PROPHET_CONFIG["add_covid_regressor"]:
        model.add_regressor(CONFIG["covid_feature_col"], standardize=False)
    return model


def fit_prophet_target(df_train_input: pd.DataFrame, target_id: str) -> Prophet:
    """Train Prophet rieng cho Revenue/COGS."""
    train_prophet = prepare_prophet_df(df_train_input, target_id)
    model = build_prophet_model()
    model.fit(train_prophet)
    return model


def make_prophet_future(dates, covid_value="auto") -> pd.DataFrame:
    """Tao future dataframe cho Prophet, co is_covid neu dung regressor."""
    future = pd.DataFrame({"ds": pd.to_datetime(pd.Index(dates))})
    if PROPHET_CONFIG["add_covid_regressor"]:
        future = add_covid_feature(future, date_col="ds", value=covid_value)
    return future


def predict_prophet_long(models: dict, dates, covid_value="auto") -> pd.DataFrame:
    """Du bao long format cho tat ca target bang Prophet."""
    frames = []
    future = make_prophet_future(dates, covid_value=covid_value)
    for uid, model in models.items():
        pred = model.predict(future)
        part = pred[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()
        part["unique_id"] = uid
        part["Prophet"] = part["yhat"].clip(lower=0)
        frames.append(part[["unique_id", "ds", "Prophet", "yhat_lower", "yhat_upper"]])
    return pd.concat(frames, ignore_index=True)


def prophet_pivot_to_wide(prophet_long: pd.DataFrame, model_col: str = "Prophet") -> pd.DataFrame:
    """Pivot ket qua Prophet sang format submission."""
    wide = (
        prophet_long
        .pivot(index="ds", columns="unique_id", values=model_col)
        .reset_index()
        .rename(columns={"ds": "Date"})
    )
    wide.columns.name = None
    wide = wide[["Date"] + CONFIG["target_cols"]]
    for col in CONFIG["target_cols"]:
        wide[col] = wide[col].clip(lower=0).round(2)
    return wide


# --- 1) Validation: train Prophet tren df_train, predict df_test ---
print("? PHUONG AN 3 - Train Prophet changepoint tren tap Train...")
prophet_eval_models = {
    uid: fit_prophet_target(df_train, uid)
    for uid in CONFIG["target_cols"]
}

prophet_eval_dates = np.sort(df_test["ds"].unique())
prophet_eval_long = predict_prophet_long(
    prophet_eval_models,
    dates=prophet_eval_dates,
    covid_value="auto",
)
prophet_eval_df = prophet_eval_long.merge(
    df_test[["unique_id", "ds", "y"]],
    on=["unique_id", "ds"],
    how="inner",
)

print("\n" + "=" * 72)
print(f"{'PHUONG AN 3 - PROPHET CHANGEPOINT VALIDATION':^72}")
print("=" * 72)
print(f"{'Metric':<18} {'Revenue':>19} {'COGS':>19} {'Avg':>12}")
print("-" * 72)

prophet_eval_results = {}
for uid in CONFIG["target_cols"]:
    mask = prophet_eval_df["unique_id"] == uid
    y_true = prophet_eval_df.loc[mask, "y"].to_numpy(dtype=float)
    y_pred = prophet_eval_df.loc[mask, "Prophet"].to_numpy(dtype=float)
    safe_y = np.where(np.abs(y_true) < 1e-8, 1e-8, y_true)
    prophet_eval_results[uid] = {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "MAPE(%)": np.mean(np.abs((y_true - y_pred) / safe_y)) * 100,
    }

for metric in ["MAE", "RMSE", "R2", "MAPE(%)"]:
    rev = prophet_eval_results["Revenue"][metric]
    cogs = prophet_eval_results["COGS"][metric]
    avg = (rev + cogs) / 2
    if metric == "R2":
        print(f"{metric:<18} {rev:>19.4f} {cogs:>19.4f} {avg:>12.4f}")
    elif metric == "MAPE(%)":
        print(f"{metric:<18} {rev:>18.2f}% {cogs:>18.2f}% {avg:>11.2f}%")
    else:
        print(f"{metric:<18} {rev:>19,.2f} {cogs:>19,.2f} {avg:>12,.2f}")
print("=" * 72)

# --- 2) Final forecast: train Prophet tren full history va xuat file rieng ---
print("\n? Train Prophet changepoint tren 100% lich su va du bao final...")
prophet_full_models = {
    uid: fit_prophet_target(df_long, uid)
    for uid in CONFIG["target_cols"]
}

prophet_future_dates = pd.date_range(CONFIG["forecast_start"], periods=CONFIG["forecast_horizon"], freq="D")
prophet_forecast_scenarios = {
    "non_covid": predict_prophet_long(prophet_full_models, prophet_future_dates, covid_value=0),
    "covid": predict_prophet_long(prophet_full_models, prophet_future_dates, covid_value=1),
    "auto": predict_prophet_long(prophet_full_models, prophet_future_dates, covid_value="auto"),
}

if CONFIG["forecast_covid_mode"] == "blend":
    p = float(np.clip(CONFIG["forecast_covid_probability"], 0.0, 1.0))
    prophet_forecast_long = prophet_forecast_scenarios["non_covid"].copy()
    prophet_forecast_long["Prophet"] = (
        (1.0 - p) * prophet_forecast_scenarios["non_covid"]["Prophet"].to_numpy(dtype=float)
        + p * prophet_forecast_scenarios["covid"]["Prophet"].to_numpy(dtype=float)
    )
    prophet_forecast_scenarios["blend"] = prophet_forecast_long
elif CONFIG["forecast_covid_mode"] in prophet_forecast_scenarios:
    prophet_forecast_long = prophet_forecast_scenarios[CONFIG["forecast_covid_mode"]]
else:
    raise ValueError("forecast_covid_mode phai la: non_covid, covid, auto, hoac blend")

prophet_submission = prophet_pivot_to_wide(prophet_forecast_long)
prophet_output_path = os.path.join(CONFIG["output_dir"], "submission_prophet_changepoint.csv")
prophet_submission.to_csv(prophet_output_path, index=False, date_format="%Y-%m-%d")

print("? Da luu Prophet changepoint submission rieng.")
print(f"   File: {prophet_output_path}")
print(f"   Changepoints: {PROPHET_CONFIG['changepoints']}")
print(f"   changepoint_prior_scale: {PROPHET_CONFIG['changepoint_prior_scale']}")
print(f"   Rows: {len(prophet_submission)} | {prophet_submission['Date'].min().date()} -> {prophet_submission['Date'].max().date()}")
display(prophet_submission.head())


## 📊 BƯỚC 6 — TRỰC QUAN HÓA DỰ BÁO

So sánh dữ liệu lịch sử với kết quả dự báo để kiểm tra tính hợp lý (sanity check).

In [ ]:
# ============================================================
# BƯỚC 6: TRỰC QUAN HÓA — LỊCH SỬ VS. DỰ BÁO
# ============================================================

# Chuẩn bị dữ liệu lịch sử ở dạng Wide để vẽ cùng trục
df_history = (
    df_long
    .pivot(index="ds", columns="unique_id", values="y")
    .reset_index()
    .rename(columns={"ds": "Date"})
)
df_history.columns.name = None

fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=False)
fig.suptitle(
    "Dự Báo Doanh Thu & Giá Vốn — MLForecast + LightGBM\n"
    f"Horizon: {CONFIG['forecast_horizon']} ngày "
    f"({CONFIG['forecast_start']} → {str(submission['Date'].max().date())})",
    fontsize=14,
    fontweight="bold",
    y=1.01,
)

plot_cfg = [
    {"col": "Revenue", "hist_color": "#1f77b4", "fcast_color": "#ff7f0e", "label": "Doanh Thu (Revenue)"},
    {"col": "COGS",    "hist_color": "#2ca02c", "fcast_color": "#d62728", "label": "Giá Vốn (COGS)"},
]

for ax, cfg in zip(axes, plot_cfg):
    col = cfg["col"]

    # Vẽ lịch sử
    ax.plot(
        df_history["Date"], df_history[col],
        color=cfg["hist_color"], linewidth=1.0,
        alpha=0.8, label="Lịch sử",
    )

    # Vẽ dự báo
    ax.plot(
        submission["Date"], submission[col],
        color=cfg["fcast_color"], linewidth=1.5,
        linestyle="--", label="Dự báo", alpha=0.9,
    )

    # Đánh dấu ranh giới lịch sử / dự báo
    ax.axvline(
        x=pd.Timestamp(CONFIG["forecast_start"]),
        color="black", linewidth=1.2, linestyle=":", alpha=0.7,
    )
    ax.text(
        pd.Timestamp(CONFIG["forecast_start"]), ax.get_ylim()[1] * 0.95,
        " Forecast →", color="black", fontsize=9, va="top",
    )

    ax.set_title(cfg["label"], fontsize=12, pad=6)
    ax.set_ylabel("Giá trị", fontsize=10)
    ax.legend(loc="upper left", fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(
    os.path.join(CONFIG["output_dir"], "forecast_visualization.png"),
    dpi=150, bbox_inches="tight",
)
plt.show()
print("✅ Đã lưu forecast_visualization.png")

## 🔍 BƯỚC 7 — PHÂN TÍCH KHẢ NĂNG GIẢI THÍCH (SHAP)

Sử dụng **SHAP TreeExplainer** để phân tích mức độ đóng góp của từng feature vào quyết định dự báo của mô hình LightGBM.

- **SHAP value dương** → Feature đẩy dự báo lên cao hơn baseline
- **SHAP value âm** → Feature kéo dự báo xuống thấp hơn baseline
- **Summary plot** thể hiện top features theo mức độ quan trọng toàn cục

In [ ]:
# ============================================================
# BƯỚC 7: PHÂN TÍCH SHAP EXPLAINABILITY
# ============================================================

def run_shap_analysis(
    fcst: MLForecast,
    df_long: pd.DataFrame,
    target_id: str,
    model_name: str,
    output_dir: str,
    output_filename: str,
    max_display: int = 20,
    sample_n: int = 2000,
) -> None:
    """
    Chạy SHAP TreeExplainer trên mô hình LightGBM đã train,
    vẽ và lưu summary_plot ra file ảnh.
    """
    print(f"⏳ Đang tính SHAP cho target: {target_id}...")

    # Lấy feature matrix qua preprocess (không có transforms parameter)
    X_full = fcst.preprocess(df_long, static_features=[])

    # Lọc chỉ lấy dữ liệu của target_id cần phân tích
    X_target = X_full[X_full["unique_id"] == target_id].copy()

    # Xác định cột feature (bỏ unique_id, ds, y)
    feature_cols = [
        c for c in X_target.columns
        if c not in ["unique_id", "ds", "y"]
    ]
    X_feat = X_target[feature_cols].dropna()

    # Lấy mẫu ngẫu nhiên để tăng tốc tính SHAP
    if len(X_feat) > sample_n:
        X_feat = X_feat.sample(n=sample_n, random_state=42)

    # Lấy mô hình LightGBM từ fcst.models_ (dict {model_name: estimator})
    raw_model = fcst.models_[model_name]

    explainer   = shap.TreeExplainer(raw_model)
    shap_values = explainer.shap_values(X_feat)

    # --- Vẽ Summary Plot ---
    fig, ax = plt.subplots(figsize=(10, 8))
    plt.sca(ax)

    shap.summary_plot(
        shap_values,
        X_feat,
        max_display=max_display,
        show=False,
        plot_size=None,
    )

    ax.set_title(
        f"SHAP Feature Importance — LightGBM ({target_id})\n"
        f"Top {max_display} features theo |SHAP value| trung bình",
        fontsize=12,
        fontweight="bold",
        pad=10,
    )
    ax.set_xlabel("SHAP value (tác động lên dự báo)", fontsize=10)

    plt.tight_layout()
    save_path = os.path.join(output_dir, output_filename)
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print(f"✅ Đã lưu SHAP summary plot: {save_path}")


# --- Chạy SHAP cho Revenue ---
run_shap_analysis(
    fcst=fcst,
    df_long=df_long_model,
    target_id="Revenue",
    model_name=CONFIG["primary_model_name"],
    output_dir=CONFIG["output_dir"],
    output_filename=CONFIG["shap_plot_filename"],
    max_display=20,
    sample_n=2000,
)

In [ ]:
# --- (Tùy chọn) Chạy SHAP cho COGS ---
run_shap_analysis(
    fcst=fcst,
    df_long=df_long_model,
    target_id="COGS",
    model_name=CONFIG["primary_model_name"],
    output_dir=CONFIG["output_dir"],
    output_filename="shap_summary_cogs.png",
    max_display=20,
    sample_n=2000,
)

## TONG KET PIPELINE

| Buoc | Mo ta | Trang thai |
|------|-------|------------|
| 1 | Khai bao CONFIG tap trung | Done |
| 2 | Doc `sales.csv`, them `is_covid`, chuyen sang Long Format | Done |
| 3 | Hoc weekly timing profile: peak week, top-week frequency, rank, proximity, one-hot 53 tuan | Done |
| 4 | Hoc yearly peak amplitude trend: recent same-week level, YoY week growth, annual peak growth | Done |
| 5 | Huan luyen MLForecast + LightGBM ensemble voi `is_covid` + `woy_*` dynamic exog | Done |
| 6 | Hoc blend weight + median bias tren validation de toi uu MAE | Done |
| 7 | Du bao cac scenario future COVID va xuat `submission.csv` | Done |

**Note tieng Viet ve phan vua thuc hien:**
- Truoc do model da biet kha tot `tuan nao trong nam co dinh`, nhung do cao cua dinh bi keo ve trung binh lich su.
- Minh da them cac cot `woy_recent_*`, `woy_same_week_growth`, `woy_projected_next_mean`, `woy_annual_peak_growth` de model nhin thay dinh 2020-2021-2022 dang tao xu huong tang.
- Cach nay khong ep nhan he so sau forecast mot cach tho bao; LightGBM tu hoc khi nao nen nang peak dua tren week-of-year va trend gan day.

**Ket qua validation sau thay doi gan nhat:**
- Revenue MAE: khoang `595,943`.
- COGS MAE: khoang `498,150`.
- Avg MAE: khoang `547,047`.

**Phan da thuc hien theo yeu cau cua ban:**
- Mat 1 - pattern/timing: model biet tuan nao trong nam co xu huong dat dinh bang `woy_peak_score`, `woy_peak_frequency`, `woy_top3_frequency`, one-hot 53 tuan va harmonic week-of-year.
- Mat 2 - gia/level cua dinh: model biet dinh cac nam gan day dang tang/giam bang `woy_recent_mean`, `woy_last_year_mean`, `woy_same_week_growth`, `woy_projected_next_mean`, `woy_annual_peak_growth`.

**Phuong an 3 - Prophet changepoint:**
- Da them nhanh Prophet rieng voi `changepoints=["2019-01-01"]` va `changepoint_prior_scale=0.1`.
- Muc dich la cho Prophet tu uon trend sau diem gay khung hoang.
- Output rieng: `submission_prophet_changepoint.csv`, dung de so sanh hoac blend them, khong ghi de `submission.csv` chinh.
